# Welcome to Pipelines!

The HuggingFace transformers library provides APIs at two different levels.

The High Level API for using open-source models for typical inference tasks is called "pipelines". It's incredibly easy to use.

You create a pipeline using something like:

`my_pipeline = pipeline("the_task_I_want_to_do")`

Followed by

`result = my_pipeline(my_input)`

And that's it!

See end of this colab for a list of all pipelines.

## My practical focus for this notebook

I am keeping the course theory intact, but I am changing the examples to match my own AI Engineering journey.

I will practice pipelines with:
- AI Engineering and LLM-related text
- my own project-style inputs
- English → Telugu translation
- zero-shot routing for AI assistant use cases
- image generation around AI Engineering

The goal is to understand the same concept deeply rather than simply copy the instructor's inputs.


## Before we start: 2 important pro-tips for using Colab:

**Pro-tip 1:**

Data Science code often gives warnings and messages. They can mostly be safely ignored! Glance over them, and if something goes wrong later, perhaps they can give you a clue.

**Pro-tip 2:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!


## A sidenote:

You may already know this, but just in case you're not familiar with the word "inference" that I use here:

When working with Data Science models, you could be carrying out 2 very different activities: **training** and **inference**.

### 1. Training  

**Training** is when you provide a model with data for it to adapt to get better at a task in the future. It does this by updating its internal settings - the parameters or weights of the model. If you're Training a model that's already had some training, the activity is called "fine-tuning".

### 2. Inference

**Inference** is when you are working with a model that has _already been trained_. You are using that model to produce new outputs on new inputs, taking advantage of everything it learned while it was being trained. Inference is also sometimes referred to as "Execution" or "Running a model".

All of our use of APIs for GPT, Claude and Gemini in the last weeks are examples of **inference**. The "P" in GPT stands for "Pre-trained", meaning that it has already been trained with data (lots of it!) In week 6 we will try fine-tuning GPT ourselves.
  
The pipelines API in HuggingFace is only for use for **inference** - running a model that has already been trained. In week 7 we will be training our own model, and we will need to use the more advanced HuggingFace APIs that we look at in the up-coming lecture.

I recorded this playlist on YouTube with more on parameters, training and inference:  
https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs


In [2]:
# Pip installs should come at the top line.
# If your Kernel ever resets, you need to run this again.

!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [3]:
# Let's check the GPU - it should be a Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

Wed Aug 26 03:09:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Imports

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


# Important Note - Hugging Face account

In Day 1, we set up a FREE account on https://huggingface.co

### If you skipped this:

Please go back and do it! Then go to the Avatar menu, Tokens, and create an API token.. And make sure it has WRITE permissions! And then add it to the secrets on the left by pressing the key button.

### If you did this (thank you!)

Click on the Key button and turn on the switch so that this notebook gets access to your Hugging Face key.

In [3]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

HF key looks good so far


## Using Pipelines from Hugging Face

A simple way to run inference for common tasks, without worrying about all the plumbing, picking reasonable defaults.


### How it works:

STEP 1: Create a pipeline - a function you can then call

```python
my_pipeline = pipeline(task, model=xx, device=xx)
```

If you don't specify a model, then Hugging Face picks one for you that's the default for the task. Specify "cuda" for the device to use an NVIDIA GPU like the one on the T4. Specify "mps" on a Mac.


STEP 2: Then call it as many times as you want:

```python
my_pipeline(input1)
my_pipeline(input2)
```

In [ ]:
# Sentiment Analysis — AI Engineering feedback
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device=0)

result = my_simple_sentiment_analyzer(
    "I am really enjoying my AI Engineering journey and learning Hugging Face pipelines!"
)
print(result)

In [ ]:
# Try another piece of feedback
result = my_simple_sentiment_analyzer(
    "The AI application is slow and the user experience is frustrating."
)
print(result)

In [ ]:
# Specify a multilingual model
better_sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=0
)

result = better_sentiment(
    "I am learning AI Engineering with Hugging Face!"
)
print(result)

In [ ]:
# Named Entity Recognition — technologies and organizations
ner = pipeline(
    "ner",
    device=0,
    aggregation_strategy="simple"
)

text = (
    "I am learning AI Engineering with Python, Hugging Face, "
    "Google Colab, Ollama, and OpenAI."
)

result = ner(text)

for entity in result:
    print(entity)

In [ ]:
# The previous NER result is available in the variable `result`
result

In [ ]:
# Question Answering with Context

question = "What am I learning?"
context = (
    "I am learning AI Engineering. "
    "I am currently exploring Hugging Face pipelines, "
    "LLMs, inference, and generative AI."
)

question_answerer = pipeline("question-answering", device=0)

result = question_answerer(
    question=question,
    context=context
)

print(result)

In [ ]:
# Text Summarization — summarize my AI Engineering notes
summarizer = pipeline("summarization", device=0)

text = """
I am learning AI Engineering by building practical projects and
understanding how models, APIs, pipelines, tokenizers, and inference
work together. Hugging Face provides access to many open-source models
and makes experimentation easier through high-level pipelines.
Google Colab gives me a convenient environment for running these models
with GPU acceleration.
"""

summary = summarizer(
    text,
    max_length=60,
    min_length=25,
    do_sample=False
)

print(summary[0]["summary_text"])

In [ ]:
# Translation — English to Spanish
translator = pipeline("translation_en_to_es", device=0)

result = translator(
    "I am learning AI Engineering and building applications with open-source models."
)

print(result[0]["translation_text"])

In [ ]:
# Translation — English to Telugu
# NLLB supports Telugu using the language code tel_Telu.

translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",
    tgt_lang="tel_Telu",
    device=0
)

result = translator(
    "I am learning AI Engineering and building applications using open-source models."
)

print(result[0]["translation_text"])

In [ ]:
# Zero-shot classification — useful for routing AI assistant requests
classifier = pipeline("zero-shot-classification", device=0)

text = "I want to learn how to deploy a fine-tuned model to production."

candidate_labels = [
    "AI Engineering",
    "Job Application",
    "Programming",
    "Travel"
]

result = classifier(
    text,
    candidate_labels=candidate_labels
)

print(result)

In [ ]:
# Text Generation
generator = pipeline("text-generation", device=0)

result = generator(
    "My goal as an AI Engineer is to",
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7
)

print(result[0]["generated_text"])

In [ ]:
# Image Generation — Diffusion Pipeline
# Personalized to my AI Engineering work.

from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
)

pipe.to("cuda")

prompt = (
    "A futuristic AI engineer building a multilingual "
    "Telugu-English AI assistant, vibrant pop-art style"
)

image = pipe(
    prompt=prompt,
    num_inference_steps=4,
    guidance_scale=0.0
).images[0]

display(image)

In [ ]:
# Audio Generation — environment check
#
# The original SpeechT5 example uses the legacy
# "matthijs/cmu-arctic-xvectors" dataset loading script.
# In my current environment, newer `datasets` versions reject
# that legacy Python dataset script.
#
# The important theory remains:
# Text -> Text-to-Speech Pipeline -> Audio
#
# First I verify my environment instead of pretending the old
# dataset-loading code still works.

import sys
import transformers
import datasets

print("Python:", sys.version)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print(
    "\nNote: the SpeechT5 model can load, but the legacy CMU Arctic "
    "xvectors dataset used in the original lesson may require an "
    "updated dataset workflow."
)

# All the available pipelines

Here are all the pipelines available from Transformers and Diffusers.

With thanks to student Lucky P for suggesting I include this!

There's a list pipelines under the Tasks on this page (you have to scroll down a bit, then expand the parameters to see the Tasks):

https://huggingface.co/docs/transformers/main_classes/pipelines

There's also this list of Tasks for Diffusion models instead of Transformers, following the image generation example where I use DiffusionPipeline above.

https://huggingface.co/docs/diffusers/en/api/pipelines/overview

If you come up with some cool examples of other pipelines, please share them with me! It's wonderful how HuggingFace makes this advanced AI functionality available for inference with such a simple API.

## My Day 2 Takeaway

### The abstraction levels I understand now

**Tokenizer** → converts text into the representation a model can process.

**Model** → performs the learned AI computation.

**`transformers.pipeline()`** → high-level interface that handles much of preprocessing, model execution, and post-processing for common inference tasks.

**`DiffusionPipeline`** → specialized complete workflow for diffusion-based generation.

### My practical workflow

`My Application → Pipeline → Preprocessing → Model → Post-processing → Output`

This is important for my AI Engineering path because I can start with the high-level Pipeline API for experimentation and later move to lower-level model/tokenizer APIs when I need more control.
